# 03 — scikit-learn End-to-End Classification

A small but complete supervised-learning workflow: split first, build preprocessing into a pipeline, compare with a dummy baseline, train logistic regression and inspect more than accuracy.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_num, y = make_classification(
    n_samples=1200, n_features=5, n_informative=3, n_redundant=1,
    weights=[0.72, 0.28], class_sep=1.0, random_state=42
)
X = pd.DataFrame(X_num, columns=['x1', 'x2', 'x3', 'x4', 'x5'])
X['segment'] = pd.cut(X['x1'], bins=[-np.inf, -0.5, 0.5, np.inf], labels=['low', 'mid', 'high'])
X.loc[X.sample(frac=0.04, random_state=7).index, 'x3'] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print('train positive rate:', y_train.mean().round(3))
print('test positive rate:', y_test.mean().round(3))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

numeric = ['x1', 'x2', 'x3', 'x4', 'x5']
categorical = ['segment']
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
])

baseline = Pipeline([
    ('prep', preprocess),
    ('model', DummyClassifier(strategy='prior')),
])
model = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=1000)),
])

baseline.fit(X_train, y_train)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score, average_precision_score, precision_score,
    recall_score, roc_auc_score
)

def evaluate(name, fitted):
    pred = fitted.predict(X_test)
    proba = fitted.predict_proba(X_test)[:, 1]
    return {
        'model': name,
        'accuracy': round(accuracy_score(y_test, pred), 3),
        'precision': round(precision_score(y_test, pred, zero_division=0), 3),
        'recall': round(recall_score(y_test, pred, zero_division=0), 3),
        'roc_auc': round(roc_auc_score(y_test, proba), 3),
        'pr_auc': round(average_precision_score(y_test, proba), 3),
    }

pd.DataFrame([evaluate('dummy_prior', baseline), evaluate('logistic_regression', model)])

## Interview points

- A **baseline** tells me whether the model adds value at all.
- The **pipeline** keeps learned preprocessing inside training instead of leaking information from the test set.
- With imbalanced classes, accuracy alone can hide poor minority-class performance, so I also inspect precision, recall, ROC-AUC and PR-AUC.
- In a real project I would decide the classification threshold from business costs/benefits on validation data rather than automatically accepting 0.5.